# Traffic Demand Prediction

## Approach
- Use Day 48 demand as the primary predictor for Day 49
- 3-way interaction features (geohash x slot x weather/roadtype/LargeVehicles/Landmarks)
- Demand lag/lead features capturing the demand curve shape
- Day 49 early (0:00-2:00) geohash-level scaling factors
- Ensemble: LightGBM + 2x CatBoost with optimised blend weights

In [ ]:
# Install dependencies
import subprocess, sys
for pkg in ['lightgbm','catboost','pygeohash','scipy']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'],check=False)

In [ ]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')
import lightgbm as lgb
from catboost import CatBoostRegressor
import pygeohash as pgh
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize
print("Libraries loaded successfully")

## 1. Load Data

In [ ]:
train = pd.read_csv('dataset/train.csv')
test  = pd.read_csv('dataset/test.csv')
train48 = train[train['day']==48].copy()
train49 = train[train['day']==49].copy()
print(f"Train48: {len(train48)} | Train49: {len(train49)} | Test: {len(test)}")
print(f"Columns: {list(train.columns)}")
train.head()

## 2. Feature Engineering

In [ ]:
def parse_ts(df):
    df=df.copy(); p=df['timestamp'].str.split(':',expand=True)
    df['hour']=p[0].astype(int); df['minute']=p[1].astype(int)
    df['time_mins']=df['hour']*60+df['minute']
    df['time_slot']=(df['time_mins']//15).astype(np.int64)
    df['hour_sin']=np.sin(2*np.pi*df['hour']/24); df['hour_cos']=np.cos(2*np.pi*df['hour']/24)
    df['time_sin']=np.sin(2*np.pi*df['time_mins']/1440); df['time_cos']=np.cos(2*np.pi*df['time_mins']/1440)
    df['is_peak']=((df['hour'].between(7,9))|(df['hour'].between(17,19))).astype(int)
    df['is_night']=((df['hour']>=22)|(df['hour']<=5)).astype(int)
    df['is_daytime']=(df['hour'].between(6,20)).astype(int)
    return df

for col in ['RoadType','LargeVehicles','Landmarks','Weather']:
    for d in [train,test,train48,train49]: d[col]=d[col].fillna('Unknown')
train=parse_ts(train); train48=parse_ts(train48); train49=parse_ts(train49); test=parse_ts(test)
print("Timestamps parsed")

In [ ]:
# Geohash decode to lat/lon
all_gh=list(set(train['geohash'].unique())|set(test['geohash'].unique()))
gh_c={}
for gh in all_gh:
    try: lat,lon=pgh.decode(gh); gh_c[gh]=(lat,lon)
    except: gh_c[gh]=(np.nan,np.nan)
gh_df=pd.DataFrame([{'geohash':g,'lat':v[0],'lon':v[1]} for g,v in gh_c.items()])
print(f"Decoded {len(gh_c)} geohashes")

In [ ]:
# Day48 feature store
gh_ts_d48=(train48.groupby(['geohash','timestamp'])['demand'].mean()
           .reset_index().rename(columns={'demand':'demand_d48'}))
gh_slot_d48=(train48.groupby(['geohash','time_slot'])['demand'].mean()
             .reset_index().rename(columns={'demand':'d48r'}))
lags={}
for k in [1,2,3,4]:
    t=gh_slot_d48.copy(); t['time_slot']+=k; lags[f'lag{k}']=t.rename(columns={'d48r':f'demand_d48_lag{k}'})
    t=gh_slot_d48.copy(); t['time_slot']-=k; lags[f'lead{k}']=t.rename(columns={'d48r':f'demand_d48_lead{k}'})

gh_stats=train48.groupby('geohash')['demand'].agg(['mean','median','std','max','min']).reset_index()
gh_stats.columns=['geohash','gh_mean','gh_med','gh_std','gh_max','gh_min']
slot_stats=train48.groupby('time_slot')['demand'].agg(['mean','median','std']).reset_index()
slot_stats.columns=['time_slot','slot_mean','slot_med','slot_std']
gh_slot=(train48.groupby(['geohash','time_slot'])['demand'].mean().reset_index().rename(columns={'demand':'gh_slot_mean'}))
gh_hour=(train48.groupby(['geohash','hour'])['demand'].mean().reset_index().rename(columns={'demand':'gh_hour_d48'}))
hr_wth=(train48.groupby(['hour','Weather'])['demand'].mean().reset_index().rename(columns={'demand':'hr_wth_mean'}))
gh_wth=(train48.groupby(['geohash','Weather'])['demand'].mean().reset_index().rename(columns={'demand':'gh_wth_mean'}))
slot_wth=(train48.groupby(['time_slot','Weather'])['demand'].mean().reset_index().rename(columns={'demand':'slot_wth_mean'}))
gh_slot_wth=(train48.groupby(['geohash','time_slot','Weather'])['demand'].mean().reset_index().rename(columns={'demand':'gh_slot_wth_mean'}))
gh_hour_wth=(train48.groupby(['geohash','hour','Weather'])['demand'].mean().reset_index().rename(columns={'demand':'gh_hour_wth_mean'}))
rl_stats=train48.groupby(['RoadType','NumberofLanes'])['demand'].agg(['mean','std']).reset_index()
rl_stats.columns=['RoadType','NumberofLanes','rl_mean','rl_std']
ts_road=(train48.groupby(['timestamp','RoadType'])['demand'].mean().reset_index().rename(columns={'demand':'ts_road_mean'}))
hr_road=(train48.groupby(['hour','RoadType'])['demand'].mean().reset_index().rename(columns={'demand':'hr_road_mean'}))
gh_rt=(train48.groupby(['geohash','RoadType'])['demand'].mean().reset_index().rename(columns={'demand':'gh_rt_mean'}))
slot_rt=(train48.groupby(['time_slot','RoadType'])['demand'].mean().reset_index().rename(columns={'demand':'slot_rt_mean'}))
gh_slot_rt=(train48.groupby(['geohash','time_slot','RoadType'])['demand'].mean().reset_index().rename(columns={'demand':'gh_slot_rt_mean'}))
hr_rl=(train48.groupby(['hour','RoadType','NumberofLanes'])['demand'].mean().reset_index().rename(columns={'demand':'hr_rl_mean'}))
gh_hour_rt=(train48.groupby(['geohash','hour','RoadType'])['demand'].mean().reset_index().rename(columns={'demand':'gh_hour_rt_mean'}))
gh_slot_lv=(train48.groupby(['geohash','time_slot','LargeVehicles'])['demand'].mean().reset_index().rename(columns={'demand':'gh_slot_lv_mean'}))
gh_slot_lm=(train48.groupby(['geohash','time_slot','Landmarks'])['demand'].mean().reset_index().rename(columns={'demand':'gh_slot_lm_mean'}))
train48['gh_p4']=train48['geohash'].str[:4]; train48['gh_p3']=train48['geohash'].str[:3]
p4=train48.groupby('gh_p4')['demand'].agg(['mean','std']).reset_index(); p4.columns=['gh_p4','p4_mean','p4_std']
p3=train48.groupby('gh_p3')['demand'].agg(['mean','std']).reset_index(); p3.columns=['gh_p3','p3_mean','p3_std']
p4_slot=(train48.groupby(['gh_p4','time_slot'])['demand'].mean().reset_index().rename(columns={'demand':'p4_slot_mean'}))
p4_slot_wth=(train48.groupby(['gh_p4','time_slot','Weather'])['demand'].mean().reset_index().rename(columns={'demand':'p4_slot_wth_mean'}))
print("Day48 feature store built")

In [ ]:
# Day49 geohash-level scaling signals
gh_d49e=train49.groupby('geohash')['demand'].agg(['mean','median','std','max']).reset_index()
gh_d49e.columns=['geohash','d49e_mean','d49e_med','d49e_std','d49e_max']
early_ts=train49['timestamp'].unique()
d48_early=train48[train48['timestamp'].isin(early_ts)].groupby('geohash')['demand'].mean().reset_index()
d48_early.columns=['geohash','gh_d48e_mean']
scale_df=gh_d49e.merge(d48_early,on='geohash',how='left')
scale_df['d49_scale']=(scale_df['d49e_mean']/(scale_df['gh_d48e_mean']+1e-8)).clip(0.05,20)
train49['gh_p4']=train49['geohash'].str[:4]
p4_d49e=train49.groupby('gh_p4')['demand'].mean().reset_index(); p4_d49e.columns=['gh_p4','p4_d49e_mean']
p4_d48e=train48[train48['timestamp'].isin(early_ts)].groupby('gh_p4')['demand'].mean().reset_index()
p4_d48e.columns=['gh_p4','p4_d48e_mean']
p4_scale=p4_d49e.merge(p4_d48e,on='gh_p4',how='left')
p4_scale['p4_d49_scale']=(p4_scale['p4_d49e_mean']/(p4_scale['p4_d48e_mean']+1e-8)).clip(0.05,20)
print("Day49 scaling signals computed")

In [ ]:
# Encoders
cat_cols=['RoadType','LargeVehicles','Landmarks','Weather']
le_dict={col:LabelEncoder().fit(pd.concat([train[col],test[col]]).unique()) for col in cat_cols}
gh_le=LabelEncoder().fit(pd.concat([train['geohash'],test['geohash']]).unique())

# Merge demand_d48 (v1 leakage strategy - key design decision)
train=train.merge(gh_ts_d48,on=['geohash','timestamp'],how='left')
test =test.merge(gh_ts_d48, on=['geohash','timestamp'],how='left')
print(f"Train with demand_d48: {train['demand_d48'].notna().sum()}/{len(train)}")
print(f"Test  with demand_d48: {test['demand_d48'].notna().sum()}/{len(test)}")

In [ ]:
def build_features(df):
    df=df.copy(); df['gh_p4']=df['geohash'].str[:4]; df['gh_p3']=df['geohash'].str[:3]
    for tbl,keys in [(gh_df,'geohash'),(gh_stats,'geohash'),(slot_stats,'time_slot'),
                     (gh_slot,['geohash','time_slot']),(gh_hour,['geohash','hour']),
                     (hr_wth,['hour','Weather']),(gh_wth,['geohash','Weather']),
                     (slot_wth,['time_slot','Weather']),(gh_slot_wth,['geohash','time_slot','Weather']),
                     (gh_hour_wth,['geohash','hour','Weather']),
                     (rl_stats,['RoadType','NumberofLanes']),(ts_road,['timestamp','RoadType']),
                     (hr_road,['hour','RoadType']),(gh_rt,['geohash','RoadType']),
                     (slot_rt,['time_slot','RoadType']),(gh_slot_rt,['geohash','time_slot','RoadType']),
                     (hr_rl,['hour','RoadType','NumberofLanes']),(gh_hour_rt,['geohash','hour','RoadType']),
                     (gh_slot_lv,['geohash','time_slot','LargeVehicles']),
                     (gh_slot_lm,['geohash','time_slot','Landmarks']),
                     (p4,'gh_p4'),(p3,'gh_p3'),
                     (p4_slot,['gh_p4','time_slot']),(p4_slot_wth,['gh_p4','time_slot','Weather']),
                     (gh_d49e,'geohash'),(scale_df[['geohash','d49_scale']],'geohash'),
                     (p4_scale[['gh_p4','p4_d49_scale']],'gh_p4')]:
        df=df.merge(tbl,on=keys,how='left')
    for k in [1,2,3,4]:
        df=df.merge(lags[f'lag{k}'],on=['geohash','time_slot'],how='left')
        df=df.merge(lags[f'lead{k}'],on=['geohash','time_slot'],how='left')
    df['d49_scale']=df['d49_scale'].fillna(df['p4_d49_scale']).fillna(1.0)
    # Hierarchical fills for sparse 3-way features
    df['gh_slot_wth_filled']=(df['gh_slot_wth_mean'].fillna(df['gh_hour_wth_mean'])
                               .fillna(df['gh_wth_mean']).fillna(df['slot_wth_mean']).fillna(df['slot_mean']))
    df['gh_slot_rt_filled']=(df['gh_slot_rt_mean'].fillna(df['gh_hour_rt_mean'])
                              .fillna(df['gh_rt_mean']).fillna(df['slot_rt_mean']).fillna(df['slot_mean']))
    df['gh_slot_lv_filled']=(df['gh_slot_lv_mean'].fillna(df['gh_rt_mean']).fillna(df['slot_mean']))
    df['gh_slot_lm_filled']=(df['gh_slot_lm_mean'].fillna(df['gh_mean']).fillna(df['slot_mean']))
    df['p4_slot_wth_filled']=(df['p4_slot_wth_mean'].fillna(df['p4_slot_mean']).fillna(df['slot_wth_mean']).fillna(df['slot_mean']))
    df['gh_slot_wth_exact']=df['gh_slot_wth_mean'].notna().astype(int)
    df['gh_slot_rt_exact']=df['gh_slot_rt_mean'].notna().astype(int)
    # Core demand features
    df['has_d48']=df['demand_d48'].notna().astype(int)
    d48f=df['gh_slot_mean'].fillna(df['p4_slot_mean']).fillna(df['slot_mean']).fillna(df['gh_mean']).fillna(0.05)
    df['demand_d48_filled']=df['demand_d48'].fillna(d48f)
    df['d48_scaled']=df['demand_d48_filled']*df['d49_scale']
    df['d48_vs_gh']=df['demand_d48_filled']/(df['gh_mean']+1e-8)
    df['d48_vs_slot']=df['demand_d48_filled']/(df['slot_mean']+1e-8)
    df['d48_vs_ghslot']=df['demand_d48_filled']/(df['gh_slot_mean'].fillna(df['p4_slot_mean']).fillna(df['slot_mean'])+1e-8)
    df['d48_vs_ghhr']=df['demand_d48_filled']/(df['gh_hour_d48']+1e-8)
    df['d49e_vs_d48']=df['d49e_mean']/(df['gh_mean']+1e-8)
    df['d49e_adj']=df['d49e_mean']*(df['slot_mean']/(df['slot_mean'].mean()+1e-8))
    df['gh_cv']=df['gh_std']/(df['gh_mean']+1e-8)
    df['d49_scaled_hr']=df['gh_hour_d48']*df['d49_scale']
    df['d48_vs_wth_fill']=df['demand_d48_filled']/(df['gh_slot_wth_filled']+1e-8)
    df['d48_vs_rt_fill']=df['demand_d48_filled']/(df['gh_slot_rt_filled']+1e-8)
    df['d48_vs_p4slot']=df['demand_d48_filled']/(df['p4_slot_mean']+1e-8)
    # Lag/Lead demand curve features
    l1=df['demand_d48_lag1'].fillna(df['demand_d48_filled']); r1=df['demand_d48_lead1'].fillna(df['demand_d48_filled'])
    l3=df['demand_d48_lag3'].fillna(df['demand_d48_filled']); r3=df['demand_d48_lead3'].fillna(df['demand_d48_filled'])
    l4=df['demand_d48_lag4'].fillna(df['demand_d48_filled']); r4=df['demand_d48_lead4'].fillna(df['demand_d48_filled'])
    df['d48_trend']=r1-l1; df['d48_accel']=r1+l1-2*df['demand_d48_filled']
    df['d48_smooth']=(l1+df['demand_d48_filled']+r1)/3
    df['d48_smooth5']=(l3+df['demand_d48_lag2'].fillna(df['demand_d48_filled'])+df['demand_d48_filled']+df['demand_d48_lead2'].fillna(df['demand_d48_filled'])+r3)/5
    df['d48_range']=r3-l3; df['d48_range4']=r4-l4
    df['d48_smooth7']=(l4+l3+l1+df['demand_d48_filled']+r1+r3+r4)/7
    df['Temperature']=df['Temperature'].fillna(train['Temperature'].median())
    df['temp_binned']=pd.cut(df['Temperature'],bins=10,labels=False)
    for col in cat_cols: df[col+'_enc']=le_dict[col].transform(df[col].astype(str))
    df['gh_enc']=gh_le.transform(df['geohash'])
    return df

train_feat=build_features(train); test_feat=build_features(test)
print("Features built successfully")

## 3. Define Features

In [ ]:
FEAT=[c for c in [
    'lat','lon','gh_enc','time_slot','time_mins','hour','minute',
    'hour_sin','hour_cos','time_sin','time_cos','is_peak','is_night','is_daytime',
    'demand_d48','demand_d48_filled','has_d48',
    'demand_d48_lag1','demand_d48_lead1','demand_d48_lag2','demand_d48_lead2',
    'demand_d48_lag3','demand_d48_lead3','demand_d48_lag4','demand_d48_lead4',
    'd48_trend','d48_accel','d48_smooth','d48_smooth5','d48_smooth7','d48_range','d48_range4',
    'd48_scaled','d48_vs_gh','d48_vs_slot','d48_vs_ghslot','d48_vs_ghhr','d48_vs_p4slot',
    'gh_mean','gh_med','gh_std','gh_max','gh_min','gh_cv',
    'slot_mean','slot_med','slot_std',
    'gh_slot_mean','gh_hour_d48','d49_scaled_hr',
    'gh_wth_mean','hr_wth_mean','slot_wth_mean','gh_slot_wth_mean','gh_hour_wth_mean',
    'gh_slot_wth_filled','gh_slot_wth_exact','p4_slot_wth_filled',
    'rl_mean','rl_std','ts_road_mean','hr_road_mean','gh_rt_mean','slot_rt_mean','hr_rl_mean',
    'gh_slot_rt_mean','gh_hour_rt_mean','gh_slot_rt_filled','gh_slot_rt_exact',
    'gh_slot_lv_mean','gh_slot_lv_filled',
    'gh_slot_lm_mean','gh_slot_lm_filled',
    'd48_vs_wth_fill','d48_vs_rt_fill',
    'p4_mean','p4_std','p3_mean','p3_std','p4_slot_mean',
    'd49e_mean','d49e_med','d49e_std','d49e_max','d49_scale','p4_d49_scale','d49e_vs_d48','d49e_adj',
    'RoadType_enc','NumberofLanes','LargeVehicles_enc','Landmarks_enc','Temperature','temp_binned','Weather_enc',
] if c in train_feat.columns]
print(f"Total features: {len(FEAT)}")
X=train_feat[FEAT].fillna(-1); y=train_feat['demand']; Xtest=test_feat[FEAT].fillna(-1)

## 4. Train Ensemble Models

In [ ]:
N=5; kf=KFold(n_splits=N,shuffle=True,random_state=42)
models_cfg=[
    ('LGB', lgb.LGBMRegressor(objective='regression',metric='rmse',verbose=-1,n_estimators=5000,
        learning_rate=0.008,max_depth=7,num_leaves=63,min_child_samples=20,feature_fraction=0.7,
        bagging_fraction=0.7,bagging_freq=5,reg_alpha=0.2,reg_lambda=1.0,random_state=42,n_jobs=-1)),
    ('CAT1',CatBoostRegressor(iterations=3000,learning_rate=0.02,depth=7,loss_function='RMSE',
        random_seed=42,verbose=0,thread_count=-1,od_type='Iter',od_wait=200,l2_leaf_reg=5,min_data_in_leaf=10)),
    ('CAT2',CatBoostRegressor(iterations=3000,learning_rate=0.015,depth=8,loss_function='RMSE',
        random_seed=123,verbose=0,thread_count=-1,od_type='Iter',od_wait=200,l2_leaf_reg=3,min_data_in_leaf=8)),
]
oofs={}; preds={}
for name,model in models_cfg:
    print(f"\n[{name}]")
    oof=np.zeros(len(X)); pred=np.zeros(len(Xtest))
    for fold,(tr_i,val_i) in enumerate(kf.split(X)):
        m=model.__class__(**model.get_params())
        if 'LGB' in name:
            m.fit(X.iloc[tr_i],y.iloc[tr_i],eval_set=[(X.iloc[val_i],y.iloc[val_i])],
                  callbacks=[lgb.early_stopping(200,verbose=False),lgb.log_evaluation(-1)])
        else:
            m.fit(X.iloc[tr_i],y.iloc[tr_i],eval_set=(X.iloc[val_i],y.iloc[val_i]),use_best_model=True,verbose=False)
        oof[val_i]=m.predict(X.iloc[val_i]); pred+=m.predict(Xtest)/N
        print(f"  Fold {fold+1}: R2={r2_score(y.iloc[val_i],oof[val_i]):.4f}")
    r2=r2_score(y,oof); print(f"  {name} OOF R2: {100*r2:.2f}")
    oofs[name]=oof; preds[name]=pred

## 5. Optimise Blend & Generate Submission

In [ ]:
ol=list(oofs.values()); pl=list(preds.values())
def neg_r2(w):
    w=np.maximum(w,0); w=w/w.sum()
    return -r2_score(y,sum(w[i]*ol[i] for i in range(3)))
res=minimize(neg_r2,[1/3,1/3,1/3],method='Nelder-Mead',options={'maxiter':2000})
w=np.maximum(res.x,0); w=w/w.sum()
for i,(n,_) in enumerate(models_cfg): print(f"{n}: {w[i]:.3f}",end='  ')
print()
blend_oof=sum(w[i]*ol[i] for i in range(3))
blend_pred=sum(w[i]*pl[i] for i in range(3))
blend_r2=r2_score(y,blend_oof)
preds_out=np.clip(blend_pred,0,1)
print(f"\nBlend OOF R2: {blend_r2:.4f} -> {100*blend_r2:.2f}")
sub=pd.DataFrame({'Index':test['Index'],'demand':preds_out})
sub.to_csv('submission.csv',index=False)
print(f"Saved submission.csv | shape={sub.shape}")
print(f"Demand stats: mean={preds_out.mean():.4f} std={preds_out.std():.4f} max={preds_out.max():.4f}")
sub.head(10)

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt
# Get LGB model from last fold (stored in m after loop if LGB is last)
# Re-train one fold for importance
m_lgb=lgb.LGBMRegressor(objective='regression',metric='rmse',verbose=-1,n_estimators=5000,
    learning_rate=0.008,max_depth=7,num_leaves=63,min_child_samples=20,feature_fraction=0.7,
    bagging_fraction=0.7,bagging_freq=5,reg_alpha=0.2,reg_lambda=1.0,random_state=42,n_jobs=-1)
tr_i,val_i=list(kf.split(X))[0]
m_lgb.fit(X.iloc[tr_i],y.iloc[tr_i],eval_set=[(X.iloc[val_i],y.iloc[val_i])],
          callbacks=[lgb.early_stopping(200,verbose=False),lgb.log_evaluation(-1)])
fi=pd.DataFrame({'feature':FEAT,'importance':m_lgb.feature_importances_}).sort_values('importance',ascending=False)
print("Top 20 Features:")
print(fi.head(20).to_string(index=False))
fig,ax=plt.subplots(figsize=(10,8))
fi.head(20).sort_values('importance').plot.barh(x='feature',y='importance',ax=ax,color='steelblue')
ax.set_title('Top 20 Feature Importances (LightGBM)'); ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()